# AI651 – Deep Learning for Space, Time and Graphs
# Assignment 2

Syed Obaidullah Hassan Chishti - 25280049

# Imports

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import copy

# Setup

## Download Dataset

In [2]:
torchvision.datasets.MovingMNIST(root='./data', split=None, download=True)

100.0%


Dataset MovingMNIST
    Number of datapoints: 10000
    Root location: ./data

## Read and split data

In [2]:
# Load the file
data = np.load("data/MovingMNIST/mnist_test_seq.npy")

data = data.astype(np.float32) / 255.0  # Normalize the data to [0, 1]

train_data = data[:, :8000, :, :]
val_data = data[:, 8000:9000, :, :]
test_data = data[:, 9000:, :, :]

print("Train data shape:", train_data.shape)
print("Validation data shape:", val_data.shape)
print("Test data shape:", test_data.shape)

Train data shape: (20, 8000, 64, 64)
Validation data shape: (20, 1000, 64, 64)
Test data shape: (20, 1000, 64, 64)


# Transformations

## Mirror

In [44]:
def mirror_horizontal(x):
    j_flipped = 63 - np.arange(x.shape[-1])
    return x[..., j_flipped]

## Rotation

In [45]:
def rotate_90(x, direction='clockwise'):
    if direction == 'clockwise':
        return torch.rot90(x, k=1, dims=(-2,-1))
    elif direction == 'counterclockwise':
        return torch.rot90(x, k=-1, dims=(-2,-1))

## Speed-up

In [ ]:
def speedup(x, k):
    new_n_frames = 5
    max_valid_index = 19

    input_indices = torch.arange(new_n_frames) * k
    input_indices = input_indices[input_indices <= max_valid_index]
    
    output_indices = torch.arange(new_n_frames, 2 * new_n_frames) * k
    output_indices = output_indices[output_indices <= max_valid_index]


    input_frames = x[input_indices]
    output_frames = x[output_indices]

    return input_frames, output_frames, input_indices, output_indices

## Checking the Code

In [47]:
# Define x
torch.manual_seed(0)
x = torch.rand(20, 1, 64, 64)

# Marker
p = torch.zeros_like(x)
p[7, 0, 10, 20] = 1

In [48]:
# (a) max|M (M (x)) − x|

x_mirrored = mirror_horizontal(x)
difference = torch.max(mirror_horizontal(x_mirrored) - x)
print("Difference after mirroring horizontally:", difference.item())

Difference after mirroring horizontally: 0.0


In [49]:
# (b) max|R4 (x) − x|

rotated_x = copy.deepcopy(x)
for i in range(4):
    rotated_x = rotate_90(rotated_x)

difference = torch.max(rotated_x - x)
print(f"Difference after rotating {90 * (i + 1)} degrees:", difference.item())

Difference after rotating 360 degrees: 0.0


In [50]:
# (c) max|M RM (x) − R−1 (x)|

first_mirror = mirror_horizontal(x)
first_rotate = rotate_90(first_mirror)
second_mirror = mirror_horizontal(first_rotate)

anticlockwise_rotated = rotate_90(x, direction='counterclockwise')

difference_combination = torch.max(second_mirror - anticlockwise_rotated)
print("Difference after combination of mirror and rotate:", difference_combination.item())

Difference after combination of mirror and rotate: 0.0


In [51]:
# (d) bright pixel of M (p) (frame, row, column)

mirrored_p = mirror_horizontal(p)
bright_pixel = torch.nonzero(mirrored_p, as_tuple=False)

if bright_pixel.numel() > 0:
    frame, row, column = bright_pixel[0][0].item(), bright_pixel[0][2].item(), bright_pixel[0][3].item()
    print(f"Bright pixel of M(p) is at: ({frame}, {row}, {column})")

Bright pixel of M(p) is at: (7, 10, 43)


In [52]:
# (e) bright pixel of R(p) (frame, row, column)

rotated_p = rotate_90(p)
bright_pixel = torch.nonzero(rotated_p, as_tuple=False)

if bright_pixel.numel() > 0:
    frame, row, column = bright_pixel[0][0].item(), bright_pixel[0][2].item(), bright_pixel[0][3].item()
    print(f"Bright pixel of R(p) is at: ({frame}, {row}, {column})")

Bright pixel of R(p) is at: (7, 43, 10)


In [85]:
# (f) 1× input / target frames

new_x_frames, output_frames, input_indices, output_indices = speedup(x, 1)

print("Input Frame Indices:", input_indices + 1) # Add 1 to input indices for 1-based indexing
print("Output Frame Indices:", output_indices + 1) # Add 1 to output indices for 1-based indexing


Input Frame Indices: tensor([1, 2, 3, 4, 5])
Output Frame Indices: tensor([ 6,  7,  8,  9, 10])


In [86]:
# (g) 2× input / target frames

new_x_frames, output_frames, input_indices, output_indices = speedup(x, 2)

print("Input Frame Indices:", input_indices + 1) # Add 1 to input indices for 1-based indexing
print("Output Frame Indices:", output_indices + 1) # Add 1 to output indices for 1-based indexing

Input Frame Indices: tensor([1, 3, 5, 7, 9])
Output Frame Indices: tensor([11, 13, 15, 17, 19])


In [87]:
# (h) 2× input / target frames

new_x_frames, output_frames, input_indices, output_indices = speedup(x, 3)

print("Input Frame Indices:", input_indices + 1) # Add 1 to input indices for 1-based indexing
print("Output Frame Indices:", output_indices + 1) # Add 1 to output indices for 1-based indexing


Input Frame Indices: tensor([ 1,  4,  7, 10, 13])
Output Frame Indices: tensor([16, 19])
